# 1. Feature Engineering Pipeline

## What is a Feature Engineering Pipeline?

A feature engineering pipeline is a sequence of steps that automatically prepares raw data for machine learning.

Instead of performing each preprocessing operation separately, we combine multiple operations into a single workflow.

A pipeline makes machine learning development more organized, consistent, reusable, and reliable.

## Why Do We Need a Pipeline?

Consider a customer churn prediction project.

The dataset contains:

- Customer age
- Tenure
- Monthly charges
- Contract type
- Internet service
- Churn status

Before training a machine learning model, we may need to:

1. Handle missing values.
2. Scale numerical features.
3. Encode categorical features.
4. Select important features.
5. Train a machine learning model.

Performing these steps manually every time can create errors.

A pipeline automates these operations.

## Basic Pipeline Workflow

Raw Dataset  
↓  
Train-Test Split  
↓  
Missing Value Handling  
↓  
Numerical Feature Scaling  
↓  
Categorical Feature Encoding  
↓  
Feature Selection  
↓  
Machine Learning Model  
↓  
Prediction and Evaluation

## Important Scikit-learn Components

### 1. Pipeline

`Pipeline` combines multiple processing steps into one workflow.

Example:

- StandardScaler
- LogisticRegression

### 2. ColumnTransformer

`ColumnTransformer` applies different preprocessing operations to different columns.

Example:

- Numerical columns → SimpleImputer + StandardScaler
- Categorical columns → SimpleImputer + OneHotEncoder

### 3. SimpleImputer

Used to replace missing values.

Common strategies:

- Mean
- Median
- Most frequent
- Constant

### 4. StandardScaler

Transforms numerical features so that they have approximately:

- Mean = 0
- Standard deviation = 1

### 5. OneHotEncoder

Converts categorical values into numerical columns.

Example:

Contract = Monthly, Yearly, Two Year

After encoding:

- Contract_Monthly
- Contract_Yearly
- Contract_Two Year

### 6. SelectKBest

Selects the most useful features based on a statistical scoring method.

### 7. Model Training

After preprocessing and feature selection, the transformed data is passed to a machine learning algorithm.

## Advantages of Feature Engineering Pipelines

1. Automates preprocessing.
2. Prevents train-test contamination.
3. Maintains consistent transformations.
4. Makes code reusable.
5. Reduces manual errors.
6. Simplifies model deployment.
7. Supports cross-validation and hyperparameter tuning.

## Important Rule

Always split the dataset before fitting preprocessing transformations.

The pipeline should learn preprocessing information from training data and apply the same transformations to test data.

# Demonstration of Feature Engineering Pipeline

## Business Scenario: Telecom Customer Churn Prediction

A telecom company wants to predict whether customers will leave its services.

The company has customer information containing numerical and categorical features.

### Sample Dataset

| CustomerID | Age | Tenure | MonthlyCharges | ContractType | InternetService | Churn |
|---|---:|---:|---:|---|---|---|
| 101 | 25 | 5 | 80 | Monthly | Fiber | Yes |
| 102 | 42 | 40 | 55 | Yearly | DSL | No |
| 103 | 31 | 12 | 95 | Monthly | Fiber | Yes |
| 104 | 50 | 60 | 45 | Two Year | DSL | No |
| 105 | 28 | 8 | 75 | Monthly | No | Yes |

### Problems in the Dataset

- Numerical features may contain missing values.
- Categorical features contain text.
- Numerical columns have different scales.
- Some features may be more useful than others.

### Our Pipeline

We will create two preprocessing branches.

**Numerical Pipeline:**
1. Replace missing values using the median.
2. Apply StandardScaler.

**Categorical Pipeline:**
1. Replace missing values using the most frequent category.
2. Apply OneHotEncoder.

**Combined Pipeline:**
1. Combine numerical and categorical transformations.
2. Select useful features.
3. Train a Logistic Regression model.
4. Evaluate the model using test data.

### Expected Result

The final pipeline should accept raw customer information and automatically perform preprocessing and prediction.

This makes the model easier to maintain and deploy.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

print("Libraries imported successfully!")
np.random.seed(42)

n = 500

data = pd.DataFrame({
    "CustomerID": range(1, n + 1),
    "Age": np.random.randint(18, 70, n),
    "Tenure": np.random.randint(1, 73, n),
    "MonthlyCharges": np.random.uniform(20, 120, n).round(2),
    "SupportCalls": np.random.randint(0, 10, n),
    "ContractType": np.random.choice(
        ["Monthly", "Yearly", "Two Year"],
        n
    ),
    "InternetService": np.random.choice(
        ["DSL", "Fiber", "No"],
        n
    )
})

# Generate churn probabilities

churn_probability = (
    0.15
    + (data["Tenure"] < 12) * 0.20
    + (data["MonthlyCharges"] > 90) * 0.15
    + (data["SupportCalls"] > 5) * 0.20
    + (data["ContractType"] == "Monthly") * 0.15
)

churn_probability = churn_probability.clip(0, 0.95)

data["Churn"] = np.random.binomial(
    1,
    churn_probability
)

# Introduce missing values for demonstration

data.loc[5:14, "Age"] = np.nan
data.loc[20:29, "MonthlyCharges"] = np.nan
data.loc[30:39, "ContractType"] = np.nan
data.loc[40:49, "InternetService"] = np.nan

print("Dataset created successfully!")

print("\nFirst Five Records:")
print(data.head())

print("\nDataset Shape:")
print(data.shape)

print("\nMissing Values:")
print(data.isnull().sum())
# CustomerID is an identifier, so we exclude it.

X = data.drop(
    columns=["CustomerID", "Churn"]
)

y = data["Churn"]

print("Feature Dataset Shape:", X.shape)
print("Target Dataset Shape:", y.shape)
numerical_features = [
    "Age",
    "Tenure",
    "MonthlyCharges",
    "SupportCalls"
]

categorical_features = [
    "ContractType",
    "InternetService"
]

print("Numerical Features:")
print(numerical_features)

print("\nCategorical Features:")
print(categorical_features)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training Features:", X_train.shape)
print("Testing Features:", X_test.shape)

print("Training Target:", y_train.shape)
print("Testing Target:", y_test.shape)
numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

print("Numerical preprocessing pipeline created!")
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

print("Categorical preprocessing pipeline created!")
preprocessor = ColumnTransformer([
    ("numerical", numerical_pipeline, numerical_features),
    ("categorical", categorical_pipeline, categorical_features)
])

print("ColumnTransformer created successfully!")
complete_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("feature_selection", SelectKBest(
        score_func=f_classif,
        k=8
    )),
    ("model", LogisticRegression(max_iter=1000))
])

print("Complete machine learning pipeline created!")
complete_pipeline.fit(
    X_train,
    y_train
)

print("Pipeline trained successfully!")
predictions = complete_pipeline.predict(X_test)

print("First 10 Predictions:")
print(predictions[:10])

print("\nActual Values:")
print(y_test.iloc[:10].values)
accuracy = accuracy_score(
    y_test,
    predictions
)

print("Model Accuracy:")
print(round(accuracy, 4))

print("\nClassification Report:")

print(classification_report(
    y_test,
    predictions,
    zero_division=0
))
print("Complete Pipeline Structure:\n")

for step_name, step_object in complete_pipeline.steps:
    print(step_name, ":", step_object)

Libraries imported successfully!
Dataset created successfully!

First Five Records:
   CustomerID   Age  Tenure  MonthlyCharges  SupportCalls ContractType  \
0           1  56.0      16           76.99             1      Monthly   
1           2  69.0      61           96.25             1       Yearly   
2           3  46.0      64          107.68             5     Two Year   
3           4  32.0      63           54.21             8      Monthly   
4           5  60.0      69          102.13             0      Monthly   

  InternetService  Churn  
0             DSL      0  
1           Fiber      1  
2           Fiber      0  
3             DSL      0  
4              No      1  

Dataset Shape:
(500, 8)

Missing Values:
CustomerID          0
Age                10
Tenure              0
MonthlyCharges     10
SupportCalls        0
ContractType       10
InternetService    10
Churn               0
dtype: int64
Feature Dataset Shape: (500, 6)
Target Dataset Shape: (500,)
Numerical Feature

# Feature Justification

## Feature Engineering and Preprocessing Justification

| Feature / Transformation | Source Columns | Logic | Reason for Creation | ML Value | Leakage Risk | Final Decision |
|---|---|---|---|---|---|---|
| Age | Age | Median imputation and scaling | Handle missing values and differences in scale | Medium | Low when fitted on training data | Retain |
| Tenure | Tenure | Median imputation and scaling | Represent customer relationship duration | High | Low when fitted on training data | Retain |
| MonthlyCharges | MonthlyCharges | Median imputation and scaling | Represent customer spending | High | Low when fitted on training data | Retain |
| SupportCalls | SupportCalls | Median imputation and scaling | Represent customer service issues | High | Low when fitted on training data | Retain |
| ContractType | ContractType | Imputation and one-hot encoding | Convert contract categories into numerical values | High | Low when fitted on training data | Retain |
| InternetService | InternetService | Imputation and one-hot encoding | Represent internet service categories | Medium | Low when fitted on training data | Retain |
| Selected Features | Preprocessed features | SelectKBest with ANOVA F-test | Keep the highest-scoring features | Helps reduce unnecessary inputs | Low when fitted on training data | Retain selected features |

## Why Do We Use Pipeline?

A pipeline combines preprocessing and model training into one workflow.

It ensures that:

- Missing values are handled consistently.
- Numerical values are scaled.
- Categorical values are encoded.
- Feature selection is performed systematically.
- The model receives properly transformed data.

## How Does the Pipeline Prevent Data Leakage?

We split the dataset before training the pipeline.

When we execute:

`complete_pipeline.fit(X_train, y_train)`

The pipeline learns:

- Median values from training data.
- Most frequent categories from training data.
- Scaling parameters from training data.
- Feature selection scores from training data.
- Model parameters from training data.

When we execute:

`complete_pipeline.predict(X_test)`

The same fitted transformations are applied to test data.

The pipeline does not learn preprocessing information from the test dataset.

## Final Learning

Feature engineering pipelines are essential for building reliable machine learning systems.

They combine data preprocessing, feature selection, and model training into a single reusable workflow.

**Final Rule:** Build preprocessing pipelines using training data and apply the same fitted pipeline to new and unseen data.